<a href="https://colab.research.google.com/github/detektor777/colab_list_image/blob/main/DiT4SR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Model to request access to:
https://huggingface.co/stabilityai/stable-diffusion-3.5-medium

Token: https://huggingface.co/settings/tokens


In [ ]:
#@title ##**Install** { display-mode: "form" }
show_log = True #@param {type:"boolean"}
model_variant = "dit4sr_f" #@param ["dit4sr_q", "dit4sr_f"]
hf_token = "" #@param {type:"string"}

import os, sys, time, subprocess

t0 = time.time()

def run(cmd, desc=None):
    """Runs a shell command. If show_log is True, the output is streamed live;
    otherwise, it is buffered and printed only on error."""
    print(f'>>> {desc or cmd}')
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if show_log:
            print(line, end='')
    p.wait()
    if p.returncode != 0:
        if not show_log:
            print(''.join(lines[-80:]))
        raise RuntimeError(f'Command exited with code {p.returncode}: {cmd}')

# --- 1. Clone the repository -----------------------------------------------
if not os.path.isdir('/content/DiT4SR'):
    run('git clone --depth 1 https://github.com/Adam-duan/DiT4SR.git /content/DiT4SR',
        'Cloning DiT4SR...')
os.chdir('/content/DiT4SR')

# --- 2. Minimal dependencies -----------------------------------------------
# IMPORTANT: Do NOT use the pins from the repository's environment.yaml! They pull old
# numpy/pillow/gradio and break Colab's pre-installed torch/torchvision/PIL
# (error "cannot import name 'is_directory' from 'PIL._util'").
# We leave Colab's torch/torchvision/numpy/Pillow UNTOUCHED.
#
# Fresh Colab instances have transformers 5.x, which requires huggingface_hub>=1.5,
# but diffusers==0.31.0 (DiT4SR pin) requires hub<1.0. Therefore, we install EVERYTHING in one
# pip call with compatible versions: diffusers 0.31 + transformers 4.46 +
# hub 0.x (to avoid "cannot import name 'is_offline_mode' from 'huggingface_hub'" error).
# transformers 4.46 is a small wheel (~10 MB), so it's fast.
# bitsandbytes is needed for 8-bit loading of T5-XXL (otherwise T4 runs out of memory).
run(f'pip install {"" if show_log else "-q"} "diffusers==0.31.0" '
    '"transformers==4.46.3" "huggingface_hub>=0.25,<1.0" "accelerate>=0.31,<1.4" '
    'sentencepiece protobuf hf_transfer bitsandbytes',
    'Installing packages (fast, torch is not reinstalled)...')

# Check that key modules can be imported
import importlib
missing = []
for m in ('torch', 'torchvision', 'PIL', 'diffusers', 'transformers', 'accelerate'):
    try:
        importlib.import_module(m)
    except Exception as e:
        missing.append(f'{m}: {e}')
import transformers as _tr
if not missing and not _tr.__version__.startswith('4.'):
    missing.append(f'transformers: old version loaded in memory {_tr.__version__}')
if missing:
    raise RuntimeError('Failed to import modules:\n' + '\n'.join(missing) +
                       '\nRestart the runtime (Runtime -> Restart session) and run the cell again.')
print(f'OK: all key modules successfully imported (transformers {_tr.__version__}).')

# --- 3. Hugging Face Token ---------------------------------------------------
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN') or ''
    except Exception:
        pass

from huggingface_hub import login, snapshot_download
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print('Successfully logged in to Hugging Face Hub.')
else:
    print('WARNING: hf_token is empty. stable-diffusion-3.5-medium is a GATED model:\n'
          '  1) accept the license: https://huggingface.co/stabilityai/stable-diffusion-3.5-medium\n'
          '  2) create a token: https://huggingface.co/settings/tokens\n'
          'Otherwise, the download below will fail with a 401/403 error.')

# Speed up weight downloading
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# Paths expected by the repository's bash/test_wollava.sh
sd35_dir = 'preset/models/stable-diffusion-3.5-medium'
dit4sr_dir = f'preset/models/{model_variant}'

# --- 4. SD3.5-medium (without transformer - it is replaced by DiT4SR, saving ~4.5 GB)
if not os.path.exists(os.path.join(sd35_dir, 'model_index.json')):
    print(f'Downloading stabilityai/stable-diffusion-3.5-medium -> {sd35_dir} (~11 GB)...')
    snapshot_download(
        repo_id='stabilityai/stable-diffusion-3.5-medium',
        local_dir=sd35_dir,
        allow_patterns=[
            'text_encoder/*', 'text_encoder_2/*', 'text_encoder_3/*',
            'tokenizer/*', 'tokenizer_2/*', 'tokenizer_3/*',
            'vae/*', 'scheduler/*', 'model_index.json',
        ],
        # '*fp16*': the main .safetensors in the SD3.5 repo are ALREADY in fp16, and the files
        # model.fp16.* are their exact duplicates (~11 GB of unnecessary traffic).
        ignore_patterns=['*.md', '*.onnx', '*openvino*', '*fp16*'],
    )
    print('Done:', sd35_dir)
else:
    print('stable-diffusion-3.5-medium is already downloaded, skipping.')

# --- 5. DiT4SR Checkpoint ------------------------------------------------------
if not os.path.isdir(os.path.join(dit4sr_dir, 'transformer')):
    print(f'Downloading {model_variant} from acceptee/DiT4SR -> {dit4sr_dir}...')
    snapshot_download(
        repo_id='acceptee/DiT4SR',
        local_dir='preset/models',
        allow_patterns=[f'{model_variant}/*'],
    )
    if not os.path.isdir(dit4sr_dir):
        raise RuntimeError(f'{dit4sr_dir} not found after download - '
                           'the structure of the acceptee/DiT4SR repository might have changed.')
    print('Done:', dit4sr_dir)
else:
    print(f'{model_variant} is already downloaded, skipping.')

print(f'\nDONE in {time.time()-t0:.0f} s. Model variant: {model_variant}')

In [ ]:
#@title ##**Upload Images** { display-mode: "form" }
import os, shutil
from google.colab import files

os.chdir('/content/DiT4SR')

upload_folder = 'inputs/user_upload'
prompt_folder = 'inputs/user_upload_prompts'
result_folder = 'results/user_upload'

for d in (upload_folder, prompt_folder, result_folder):
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

uploaded = files.upload()
for filename in uploaded.keys():
    dst = os.path.join(upload_folder, filename)
    shutil.move(filename, dst)
    print(f'{filename} -> {dst}')

print(f'\n{len(uploaded)} images uploaded to {upload_folder}')

In [ ]:
#@title ##**Prompt (instead of LLaVA)** { display-mode: "form" }
# DiT4SR expects one .txt file with a description for each image
# (originally they are generated by LLaVA). We save your prompt to a .txt file
# for each uploaded image. A specific description of the image content
# (e.g., "a portrait of a young woman with curly red hair, studio lighting")
# usually yields better results than a generic prompt.

prompt = "hyperrealism" #@param {type:"string"}

import os

if 'upload_folder' not in globals():
    raise RuntimeError('Please run the "Upload Images" cell first.')
if not os.path.isdir(upload_folder) or not os.listdir(upload_folder):
    raise RuntimeError(f'No images found in {upload_folder}. Please run "Upload Images".')

os.makedirs(prompt_folder, exist_ok=True)
count = 0
for fname in sorted(os.listdir(upload_folder)):
    base, _ = os.path.splitext(fname)
    with open(os.path.join(prompt_folder, base + '.txt'), 'w', encoding='utf-8') as f:
        f.write(prompt)
    count += 1

print(f'Prompt saved to {count} files in {prompt_folder}:')
print(f'  "{prompt}"')

In [ ]:
#@title ##**Run** { display-mode: "form" }
show_log = True #@param {type:"boolean"}
num_inference_steps = 45 #@param {type:"slider", min:0, max:200, step:1}
upscale = 2 #@param [1, 2, 3, 4] {type:"raw"}
align_method = "adain" #@param ["adain", "wavelet", "nofix"]
seed = -1 #@param {type:"integer"}
t5_mode = "4bit" #@param ["4bit", "8bit", "fp16"]
vae_encoder_tiled_size = 512 #@param [256, 384, 512, 768, 1024] {type:"raw"}
vae_decoder_tiled_size = 96 #@param [64, 96, 128, 160, 224] {type:"raw"}
latent_tiled_size = 96 #@param [48, 64, 96, 128] {type:"raw"}
latent_tiled_overlap = 24 #@param [8, 16, 24, 32] {type:"raw"}
extra_args = "" #@param {type:"string"}
# seed: -1 = pick a new random seed on every run (default). Set a fixed
# non-negative number here to reproduce the same result across runs.

# t5_mode: precision of the T5-XXL text encoder. VRAM budget on T4 = 14.6 GiB, of which
# the DiT4SR transformer takes 6.2 + VAE/CLIP ~1.6 = ~7.8 GiB. So:
#   "4bit" (~2.6 GiB) - FOR T4, total ~10.4 GiB + headroom for inference. Default.
#   "8bit" (~4.9 GiB) - tight on T4, reliable on L4.
#   "fp16" (~9.5 GiB) - A100 only (40 GB). Will CUDA OOM on T4!
# T5 only encodes the prompt text - quantization has minimal impact on image quality.
#
# vae_encoder_tiled_size / vae_decoder_tiled_size: tile sizes for the tiled VAE
# (encoder in pixels, decoder in latent units). The repo defaults are sized for a
# 24 GB GPU; the values above are safe defaults for a T4. Larger tiles = faster but
# more VRAM; smaller tiles = slower but safer against CUDA OOM. Quality is unaffected -
# tiles are stitched together with overlap.
#
# latent_tiled_size / latent_tiled_overlap: tile size and overlap (in latent units)
# for the tiled diffusion sampling loop itself (separate from the tiled VAE above).
# Same trade-off: smaller tiles use less VRAM during the denoising steps but are
# slower and can show faint seams if overlap is too small; larger overlap blends
# tiles more smoothly at the cost of extra compute.

import os, sys, gc, time, subprocess, random
import torch

os.chdir('/content/DiT4SR')
t0 = time.time()

if seed == -1:
    seed = random.randint(0, 2**31 - 1)
    print(f'Random seed picked for this run: {seed}\n')

for v in ('upload_folder', 'prompt_folder', 'result_folder', 'sd35_dir', 'dit4sr_dir'):
    if v not in globals():
        raise RuntimeError(f'"{v}" is not defined - run the Install, Upload Images and Prompt cells first.')

if not os.path.exists('test/test_wollava.py'):
    raise RuntimeError('test/test_wollava.py not found - the repository structure may have changed.')

# ---------------------------------------------------------------------------
# IMPORTANT: the original test_wollava.py loads models WITHOUT torch_dtype, so
# T5-XXL is created in fp32 (~19 GB) in system RAM. Colab's T4 only has ~12.7 GB
# RAM -> the process got killed by the kernel (SIGKILL, code -9). The colab_run.py
# wrapper below only swaps in a memory-efficient model loader (fp16 + straight to
# GPU, T5 optionally quantized) and calls the original main() unchanged.
# ---------------------------------------------------------------------------
SCRIPT = r'''
import os, sys, argparse, importlib.util
sys.path.append(os.getcwd())
import torch

def load_module():
    spec = importlib.util.spec_from_file_location('test_wollava', 'test/test_wollava.py')
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

def make_loader(t5_mode):
    def load_pipeline_lowmem(args, accelerator):
        from diffusers import AutoencoderKL, FlowMatchEulerDiscreteScheduler
        from transformers import (CLIPTokenizer, T5TokenizerFast,
                                  CLIPTextModelWithProjection, T5EncoderModel)
        from model_dit4sr.transformer_sd3 import SD3Transformer2DModel
        from pipelines.pipeline_dit4sr import StableDiffusion3ControlNetPipeline

        device = accelerator.device
        dtype = torch.float16
        base = args.pretrained_model_name_or_path

        def log(msg):
            print('[lowmem] ' + msg, flush=True)

        log('scheduler + tokenizers...')
        scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(base, subfolder='scheduler')
        tok1 = CLIPTokenizer.from_pretrained(base, subfolder='tokenizer')
        tok2 = CLIPTokenizer.from_pretrained(base, subfolder='tokenizer_2')
        tok3 = T5TokenizerFast.from_pretrained(base, subfolder='tokenizer_3')

        def vram(tag):
            free, total = torch.cuda.mem_get_info()
            log('%s: VRAM used %.1f / %.1f GiB' % (tag, (total - free) / 1024**3, total / 1024**3))

        # Load T5-XXL FIRST, while the GPU is still empty: bitsandbytes quantization
        # needs some temporary headroom. (T5 used to load last, after the transformer
        # had already taken 6.2 GiB of VRAM - and it would CUDA OOM.)
        if t5_mode in ('4bit', '8bit'):
            from transformers import BitsAndBytesConfig
            if t5_mode == '4bit':
                log('T5-XXL (4-bit NF4, ~2.6 GiB VRAM) -> GPU...')
                qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                          bnb_4bit_use_double_quant=True,
                                          bnb_4bit_compute_dtype=dtype)
            else:
                log('T5-XXL (8-bit, ~4.9 GiB VRAM) -> GPU...')
                qcfg = BitsAndBytesConfig(load_in_8bit=True)
            te3 = T5EncoderModel.from_pretrained(
                base, subfolder='text_encoder_3',
                quantization_config=qcfg, device_map={'': 0}, torch_dtype=dtype)
        else:
            log('T5-XXL (fp16, ~9.5 GiB VRAM, A100 only!) -> GPU...')
            te3 = T5EncoderModel.from_pretrained(
                base, subfolder='text_encoder_3', torch_dtype=dtype,
                low_cpu_mem_usage=True).to(device)
        torch.cuda.empty_cache()
        vram('after T5')

        log('VAE (fp16) -> GPU...')
        vae = AutoencoderKL.from_pretrained(base, subfolder='vae', torch_dtype=dtype).to(device)

        log('CLIP text encoders (fp16) -> GPU...')
        te1 = CLIPTextModelWithProjection.from_pretrained(
            base, subfolder='text_encoder', torch_dtype=dtype,
            low_cpu_mem_usage=True).to(device)
        te2 = CLIPTextModelWithProjection.from_pretrained(
            base, subfolder='text_encoder_2', torch_dtype=dtype,
            low_cpu_mem_usage=True).to(device)

        log('DiT4SR transformer (fp16, 6.2 GiB) -> GPU...')
        transformer = SD3Transformer2DModel.from_pretrained(
            args.transformer_model_name_or_path, subfolder='transformer',
            torch_dtype=dtype, low_cpu_mem_usage=True).to(device)
        torch.cuda.empty_cache()

        for m in (vae, transformer, te1, te2, te3):
            m.requires_grad_(False)

        pipe = StableDiffusion3ControlNetPipeline(
            vae=vae, text_encoder=te1, text_encoder_2=te2, text_encoder_3=te3,
            tokenizer=tok1, tokenizer_2=tok2, tokenizer_3=tok3,
            transformer=transformer, scheduler=scheduler)

        # 1) TILED VAE: test_wollava.py does NOT enable it, so the VAE would encode
        #    a 2048x2048 image in one shot and CUDA OOM. We force it on here.
        pipe._init_tiled_vae(
            encoder_tile_size=args.vae_encoder_tiled_size,
            decoder_tile_size=args.vae_decoder_tiled_size)
        log('tiled VAE enabled (encoder %d px, decoder %d latent)' %
            (args.vae_encoder_tiled_size, args.vae_decoder_tiled_size))

        # 2) The CLIP encoders (~1.6 GiB) are only needed for a couple of seconds to
        #    encode the prompt. Keep them in RAM and move to GPU only inside encode_prompt.
        te1.to('cpu'); te2.to('cpu')
        torch.cuda.empty_cache()
        _orig_encode_prompt = pipe.encode_prompt
        def _shuttle_encode_prompt(*a, **kw):
            te1.to(device); te2.to(device)
            try:
                return _orig_encode_prompt(*a, **kw)
            finally:
                te1.to('cpu'); te2.to('cpu')
                torch.cuda.empty_cache()
        pipe.encode_prompt = _shuttle_encode_prompt

        vram('models loaded')
        return pipe
    return load_pipeline_lowmem

def build_args():
    p = argparse.ArgumentParser()
    p.add_argument('--pretrained_model_name_or_path', type=str, default='preset/models/stable-diffusion-3.5-medium')
    p.add_argument('--transformer_model_name_or_path', type=str, default='preset/models/dit4sr_q')
    p.add_argument('--prompt', type=str, default='')
    p.add_argument('--added_prompt', type=str, default='Cinematic, hyper sharpness, highly detailed, perfect without deformations, camera, hyper detailed photo - realistic maximum detail, 32k, Color Grading, ultra HD, extreme meticulous detailing, skin pore detailing. ')
    p.add_argument('--negative_prompt', type=str, default='motion blur, noisy, dotted, bokeh, pointed, CG Style, 3D render, unreal engine, blurring, dirty, messy, worst quality, low quality, frames, watermark, signature, jpeg artifacts, deformed, lowres, chaotic')
    p.add_argument('--image_path', type=str, default=None)
    p.add_argument('--prompt_path', type=str, default='prompt_LR')
    p.add_argument('--output_dir', type=str, default='results')
    p.add_argument('--mixed_precision', type=str, default='fp16')
    p.add_argument('--guidance_scale', type=float, default=8.0)
    p.add_argument('--num_inference_steps', type=int, default=40)
    p.add_argument('--process_size', type=int, default=512)
    p.add_argument('--vae_decoder_tiled_size', type=int, default=224)
    p.add_argument('--vae_encoder_tiled_size', type=int, default=1024)
    p.add_argument('--latent_tiled_size', type=int, default=64)
    p.add_argument('--latent_tiled_overlap', type=int, default=24)
    p.add_argument('--upscale', type=int, default=4)
    p.add_argument('--seed', type=int, default=None)
    p.add_argument('--sample_times', type=int, default=1)
    p.add_argument('--align_method', type=str, choices=['wavelet', 'adain', 'nofix'], default='adain')
    p.add_argument('--start_point', type=str, choices=['lr', 'noise'], default='noise')
    p.add_argument('--save_prompts', action='store_true')
    p.add_argument('--revision', type=str, default=None)
    p.add_argument('--variant', type=str, default=None)
    p.add_argument('--t5_mode', type=str, choices=['4bit', '8bit', 'fp16'], default='4bit')
    return p.parse_args()

if __name__ == '__main__':
    args = build_args()
    tw = load_module()
    tw.load_dit4sr_pipeline = make_loader(args.t5_mode)
    tw.main(args)
'''

with open('colab_run.py', 'w', encoding='utf-8') as f:
    f.write(SCRIPT)

# '-u' = unbuffered output, so progress prints immediately
cmd = [
    sys.executable, '-u', 'colab_run.py',
    '--pretrained_model_name_or_path', sd35_dir,
    '--transformer_model_name_or_path', dit4sr_dir,
    '--image_path', upload_folder,
    '--prompt_path', prompt_folder,
    '--output_dir', result_folder,
    '--mixed_precision', 'fp16',
    '--num_inference_steps', str(num_inference_steps),
    '--upscale', str(upscale),
    '--align_method', align_method,
    '--seed', str(seed),
    '--t5_mode', t5_mode,
    '--vae_encoder_tiled_size', str(vae_encoder_tiled_size),
    '--vae_decoder_tiled_size', str(vae_decoder_tiled_size),
    '--latent_tiled_size', str(latent_tiled_size),
    '--latent_tiled_overlap', str(latent_tiled_overlap),
] + (extra_args.split() if extra_args else [])

print('Command:', ' '.join(cmd), '\n')

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'GPU: {torch.cuda.get_device_name(0)}, '
          f'{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GiB VRAM')
else:
    print('WARNING: no GPU detected! Runtime -> Change runtime type -> T4 GPU.')

# ---------------------------------------------------------------------------
# IMPORTANT: subprocess.Popen(..., text=True) enables "universal newlines" -
# Python automatically translates EVERY '\r' in the stream into '\n'. But tqdm
# (progress bars for shard loading, tiled VAE, inference steps) updates its line
# via '\r', never emitting '\n'. Because of this hidden translation, every single
# progress-bar refresh used to turn into a separate "real" line ending in '\n' ->
# the log filled up with dozens of duplicate lines like "50%|...", "51%|...",
# "52%|..." instead of one line updating in place.
#
# Fix: read the process's stdout as RAW BYTES (no text mode, no newline
# translation) and split it ourselves on '\r' and '\n':
#   '\r' -> progress bar updates in place (print(..., end='\r'))
#   '\n' -> line is finished, move to a new line
# This reproduces normal terminal behavior.
# ---------------------------------------------------------------------------
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)

KEYWORDS = ('process', 'inference time', 'input size', 'Error', 'error')
captured = []
raw = b''

def _handle_line(text, sep):
    """Print and log a single line of output. sep = b'\\r' or b'\\n'."""
    if not text.strip():
        return
    captured.append(text + '\n')
    if show_log:
        # progress bars ('\r') overwrite in place, regular lines ('\n') print as usual
        print(text, end=('\r' if sep == b'\r' else '\n'), flush=True)
    elif any(k in text for k in KEYWORDS):
        print(text, flush=True)

while True:
    chunk = process.stdout.read(4096)
    if not chunk:
        if process.poll() is not None:
            break
        continue
    raw += chunk
    while True:
        idx_r = raw.find(b'\r')
        idx_n = raw.find(b'\n')
        candidates = [i for i in (idx_r, idx_n) if i != -1]
        if not candidates:
            break
        idx = min(candidates)
        sep = raw[idx:idx + 1]
        line_bytes = raw[:idx]
        raw = raw[idx + 1:]
        _handle_line(line_bytes.decode('utf-8', errors='replace'), sep)

# leftover without a trailing \r / \n (if the process ended mid-line)
if raw.strip():
    _handle_line(raw.decode('utf-8', errors='replace'), b'\n')

if show_log:
    print()  # make sure there's a newline after the last progress bar

process.wait()

# The script saves results to {result_folder}/sample00/
out_images = []
for root, _, fnames in os.walk(result_folder):
    for f in fnames:
        if os.path.splitext(f.lower())[1] in {'.png', '.jpg', '.jpeg', '.webp'}:
            out_images.append(os.path.join(root, f))

if process.returncode != 0:
    print(f'\nERROR: script exited with code {process.returncode}')
    print('--- Last log lines ---')
    print(''.join(captured[-60:]))
    if process.returncode == -9:
        print('\nCode -9 (SIGKILL) = out of SYSTEM RAM. Make sure t5_mode="8bit",\n'
              'then restart the runtime (Runtime -> Restart session) to free memory,\n'
              'and re-run the Install -> Upload -> Prompt -> Run cells.')
    if any('out of memory' in l.lower() for l in captured):
        print('\nCUDA OOM (out of GPU memory):\n'
              '1) Make sure t5_mode="4bit" (required for T4)\n'
              '2) Restart the runtime (Runtime -> Restart session) to free VRAM\n'
              '3) Lower the input image resolution or set upscale=2\n'
              '4) Lower vae_encoder_tiled_size / vae_decoder_tiled_size and/or\n'
              '   latent_tiled_size above (e.g. latent_tiled_size=48)')
elif not out_images:
    print('\nERROR: the script finished but produced no output images.')
    print(''.join(captured[-60:]))
else:
    print(f'\nDONE: {len(out_images)} image(s) -> {result_folder}')
    for f in out_images:
        print('  ->', f)

print(f'\nElapsed time: {time.time()-t0:.1f} s')

In [ ]:
#@title ##**Visualization** { display-mode: "form" }
import os, base64
from io import BytesIO
import PIL.Image
from IPython.display import HTML, display

os.chdir('/content/DiT4SR')

if 'upload_folder' not in globals() or 'result_folder' not in globals():
    raise RuntimeError('Сначала запустите ячейки "Upload Images" и "Run".')

IMG_EXT = {'.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tiff'}

# Расширения, которые canvas.toDataURL умеет кодировать напрямую в браузере.
# Остальные (bmp, tiff, ...) при скачивании будут сохранены как PNG.
CANVAS_MIME = {
    '.png': 'image/png',
    '.jpg': 'image/jpeg',
    '.jpeg': 'image/jpeg',
    '.webp': 'image/webp',
}

def is_image(f):
    return os.path.splitext(f.lower())[1] in IMG_EXT

def image_to_base64(image):
    buffered = BytesIO()
    image.save(buffered, format='PNG')
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

def resize_image_maintain_aspect(image, max_width, target_height=None):
    width, height = image.size
    if width > max_width:
        new_height = int(height * max_width / width)
        image = image.resize((max_width, new_height), PIL.Image.Resampling.LANCZOS)
    if target_height is not None and image.size[1] != target_height:
        new_width = int(image.size[0] * target_height / image.size[1])
        image = image.resize((new_width, target_height), PIL.Image.Resampling.LANCZOS)
    return image

def find_images(root):
    found = []
    for dirpath, _, fnames in os.walk(root):
        for fname in sorted(fnames):
            if is_image(fname):
                found.append(os.path.join(dirpath, fname))
    return sorted(found)

# 1. Входные файлы, сгруппированные по базовому имени (при дублях приоритет .png)
filenames_input_all = [f for f in os.listdir(upload_folder) if is_image(f)]
input_by_base = {}
for f in filenames_input_all:
    base, ext = os.path.splitext(f)
    base_lower, ext_lower = base.lower(), ext.lower()
    if base_lower not in input_by_base:
        input_by_base[base_lower] = f
    elif ext_lower == '.png':
        input_by_base[base_lower] = f

# 2. Выходные файлы (результаты могут лежать в подпапках sample00/, sample01/, ...)
paths_output_all = find_images(result_folder)

# 3. Сопоставление пар до/после по базовому имени
matched_pairs = []
for path_output in paths_output_all:
    out_filename = os.path.basename(path_output)
    out_base_lower = os.path.splitext(out_filename)[0].lower()
    if out_base_lower in input_by_base:
        matched_pairs.append((input_by_base[out_base_lower], path_output))

if not matched_pairs:
    print(f'Пары до/после не найдены между {upload_folder} и {result_folder}.')
else:
    for idx, (filename, path_output) in enumerate(matched_pairs):
        try:
            image_before = PIL.Image.open(os.path.join(upload_folder, filename))
            image_after = PIL.Image.open(path_output)

            max_width = min(image_after.size[0], 1000)
            image_after = resize_image_maintain_aspect(image_after, max_width)
            target_height = image_after.size[1]
            image_before = resize_image_maintain_aspect(image_before, max_width, target_height)

            if image_before.mode != 'RGB':
                image_before = image_before.convert('RGB')
            if image_after.mode != 'RGB':
                image_after = image_after.convert('RGB')

            before_base64 = image_to_base64(image_before)
            after_base64 = image_to_base64(image_after)

            # Формат, в котором изначально был результат — используем его при скачивании,
            # если браузер умеет такое кодировать через canvas, иначе откатываемся на PNG.
            out_ext = os.path.splitext(path_output)[1].lower()
            download_mime = CANVAS_MIME.get(out_ext, 'image/png')
            download_ext = out_ext if out_ext in CANVAS_MIME else '.png'

            uid = f"cmp{idx}"
            base_name = os.path.splitext(filename)[0]

            html_code = f"""
            <style>
                .download-btn-{uid} {{
                    background-color: #28a745;
                    color: white;
                    border: none;
                    border-radius: 4px;
                    padding: 5px 10px;
                    cursor: pointer;
                    display: flex;
                    align-items: center;
                    gap: 6px;
                    font-family: sans-serif;
                    font-size: 12px;
                    font-weight: bold;
                    flex-shrink: 0;
                    transition: background-color 0.15s ease-in-out;
                }}
                .download-btn-{uid}:hover {{
                    background-color: #218838;
                }}
                .download-btn-{uid}:active {{
                    background-color: #1e7e34;
                }}
            </style>
            <div style="width: {image_after.size[0]}px; margin-bottom: 30px;">
                <div style="margin-bottom:6px; font-family: sans-serif; font-size: 13px; font-weight: bold;">
                    {filename} &rarr; {os.path.basename(path_output)}
                </div>
                <div id="{uid}" style="position: relative; width: 100%; height: {image_after.size[1]}px;">
                    <div style="position: relative; width: 100%; height: 100%; overflow: hidden;">
                        <img class="img-before" src="data:image/png;base64,{before_base64}" style="position: absolute; width: 100%; height: 100%;">
                        <div class="clip-wrap" style="position: absolute; width: 100%; height: 100%; overflow: hidden; clip-path: inset(0 0 0 50%);">
                            <canvas class="canvas-after" style="position: absolute; width: 100%; height: 100%; opacity: 1;"></canvas>
                        </div>
                    </div>
                    <div class="slider" style="position: absolute; top: 0; bottom: 0; width: 4px; background: white; cursor: ew-resize; left: 50%; box-shadow: 0 0 5px rgba(0,0,0,0.5);">
                        <div style="position: absolute; top: 50%; transform: translateY(-50%); width: 20px; height: 20px; background: white; border-radius: 50%; left: -8px;"></div>
                    </div>
                    <div style="position: absolute; top: 8px; left: 8px; color: white; font-family: sans-serif; font-size: 14px; text-shadow: 0 1px 3px rgba(0,0,0,0.8);">Before</div>
                    <div style="position: absolute; top: 8px; right: 8px; color: white; font-family: sans-serif; font-size: 14px; text-shadow: 0 1px 3px rgba(0,0,0,0.8);">After</div>
                </div>

                <div style="display: flex; flex-direction: column; gap: 8px; margin-top: 8px; font-family: sans-serif; font-size: 13px; color: inherit; min-width: 360px;">
                    <div style="display: flex; align-items: center; gap: 10px;">
                        <span style="white-space: nowrap; flex-shrink: 0; min-width: 90px;">Blend original:</span>
                        <input type="range" class="blend-slider" min="0" max="100" value="100" step="1" style="flex: 1;">
                        <span class="blend-value" style="min-width: 45px; text-align: right; flex-shrink: 0;">100%</span>
                        <div style="width: 100px; flex-shrink: 0;"></div>
                    </div>

                    <div style="display: flex; align-items: center; gap: 10px;">
                        <span style="white-space: nowrap; flex-shrink: 0; min-width: 90px;">Sharpness:</span>
                        <input type="range" class="sharpness-slider" min="0.0" max="7.0" value="0.0" step="0.1" style="flex: 1;">
                        <span class="sharpness-value" style="min-width: 45px; text-align: right; flex-shrink: 0;">0.0</span>
                        <button class="download-btn-{uid}" title="Download blended and sharpened image ({download_ext})" style="width: 100px;">
                            <svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" style="display: inline-block;">
                                <path d="M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4"></path>
                                <polyline points="7 10 12 15 17 10"></polyline>
                                <line x1="12" y1="15" x2="12" y2="3"></line>
                            </svg>
                            Download
                        </button>
                    </div>
                </div>
            </div>
            <script>
                (function() {{
                    const root = document.getElementById('{uid}');
                    const slider = root.querySelector('.slider');
                    const container = root.querySelector('div:nth-child(2)');
                    const clipDiv = root.querySelector('.clip-wrap');
                    let isDragging = false;

                    slider.addEventListener('mousedown', (e) => {{ isDragging = true; e.preventDefault(); }});
                    document.addEventListener('mouseup', () => {{ isDragging = false; }});
                    document.addEventListener('mousemove', (e) => {{
                        if (!isDragging) return;
                        const rect = container.getBoundingClientRect();
                        let x = e.clientX - rect.left;
                        if (x < 0) x = 0;
                        if (x > rect.width) x = rect.width;
                        const percentage = (x / rect.width) * 100;
                        slider.style.left = percentage + '%';
                        clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                    }});

                    slider.addEventListener('touchstart', (e) => {{ isDragging = true; e.preventDefault(); }});
                    document.addEventListener('touchend', () => {{ isDragging = false; }});
                    document.addEventListener('touchmove', (e) => {{
                        if (!isDragging) return;
                        const rect = container.getBoundingClientRect();
                        let x = e.touches[0].clientX - rect.left;
                        if (x < 0) x = 0;
                        if (x > rect.width) x = rect.width;
                        const percentage = (x / rect.width) * 100;
                        slider.style.left = percentage + '%';
                        clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                    }});
                }})();

                (function() {{
                    const root = document.getElementById('{uid}');
                    const wrapper = root.parentElement;
                    const blendSlider = wrapper.querySelector('.blend-slider');
                    const blendValue = wrapper.querySelector('.blend-value');
                    const sharpnessSlider = wrapper.querySelector('.sharpness-slider');
                    const sharpnessValue = wrapper.querySelector('.sharpness-value');

                    const canvasAfter = root.querySelector('.canvas-after');
                    const downloadBtn = wrapper.querySelector('.download-btn-{uid}');

                    const downloadMime = "{download_mime}";
                    const downloadExt = "{download_ext}";

                    const imgBeforeSource = new Image();
                    imgBeforeSource.src = "data:image/png;base64,{before_base64}";

                    const imgAfterSource = new Image();
                    imgAfterSource.src = "data:image/png;base64,{after_base64}";

                    let originalImgData = null;
                    let smoothedPixels = null;
                    let ctx = null;

                    function getSmoothedPixels(imgData) {{
                        const w = imgData.width;
                        const h = imgData.height;
                        const src = imgData.data;
                        const dst = new Uint8ClampedArray(src.length);
                        dst.set(src);

                        const kernel = [
                            1, 1, 1,
                            1, 5, 1,
                            1, 1, 1
                        ];
                        const divisor = 13;

                        for (let y = 1; y < h - 1; y++) {{
                            for (let x = 1; x < w - 1; x++) {{
                                let r = 0, g = 0, b = 0;
                                for (let ky = -1; ky <= 1; ky++) {{
                                    const rowOffset = (y + ky) * w * 4;
                                    for (let kx = -1; kx <= 1; kx++) {{
                                        const idx = rowOffset + (x + kx) * 4;
                                        const weight = kernel[(ky + 1) * 3 + (kx + 1)];
                                        r += src[idx] * weight;
                                        g += src[idx + 1] * weight;
                                        b += src[idx + 2] * weight;
                                    }}
                                }}
                                const dstIdx = (y * w + x) * 4;
                                dst[dstIdx]     = Math.min(255, Math.max(0, r / divisor));
                                dst[dstIdx + 1] = Math.min(255, Math.max(0, g / divisor));
                                dst[dstIdx + 2] = Math.min(255, Math.max(0, b / divisor));
                            }}
                        }}
                        return dst;
                    }}

                    function applySharpness(factor) {{
                        if (!originalImgData || !smoothedPixels) return;

                        const w = canvasAfter.width;
                        const h = canvasAfter.height;

                        if (factor === 1.0) {{
                            ctx.putImageData(originalImgData, 0, 0);
                            return;
                        }}

                        const src = originalImgData.data;
                        const smoothed = smoothedPixels;

                        const outputData = ctx.createImageData(w, h);
                        const dst = outputData.data;

                        for (let i = 0; i < src.length; i += 4) {{
                            dst[i]     = Math.min(255, Math.max(0, smoothed[i] * (1 - factor) + src[i] * factor));
                            dst[i + 1] = Math.min(255, Math.max(0, smoothed[i + 1] * (1 - factor) + src[i + 1] * factor));
                            dst[i + 2] = Math.min(255, Math.max(0, smoothed[i + 2] * (1 - factor) + src[i + 2] * factor));
                            dst[i + 3] = src[i + 3];
                        }}
                        ctx.putImageData(outputData, 0, 0);
                    }}

                    imgAfterSource.onload = () => {{
                        canvasAfter.width = imgAfterSource.width;
                        canvasAfter.height = imgAfterSource.height;
                        ctx = canvasAfter.getContext('2d');

                        ctx.drawImage(imgAfterSource, 0, 0);
                        originalImgData = ctx.getImageData(0, 0, canvasAfter.width, canvasAfter.height);

                        smoothedPixels = getSmoothedPixels(originalImgData);

                        applySharpness(0.0);
                    }};

                    sharpnessSlider.addEventListener('input', (e) => {{
                        const val = parseFloat(e.target.value);
                        sharpnessValue.textContent = val.toFixed(1);
                        applySharpness(val);
                    }});

                    blendSlider.addEventListener('input', (e) => {{
                        const val = e.target.value;
                        blendValue.textContent = val + '%';
                        canvasAfter.style.opacity = val / 100;
                    }});

                    downloadBtn.addEventListener('click', () => {{
                        const canvas = document.createElement('canvas');
                        const w = canvasAfter.width;
                        const h = canvasAfter.height;
                        canvas.width = w;
                        canvas.height = h;

                        const downloadCtx = canvas.getContext('2d');

                        downloadCtx.drawImage(imgBeforeSource, 0, 0, w, h);

                        const opacity = parseFloat(blendSlider.value) / 100;
                        downloadCtx.globalAlpha = opacity;
                        downloadCtx.drawImage(canvasAfter, 0, 0, w, h);

                        const link = document.createElement('a');
                        link.download = 'blended_{base_name}' + downloadExt;
                        link.href = canvas.toDataURL(downloadMime, 0.95);
                        link.click();
                    }});
                }})();
            </script>
            """

            display(HTML(html_code))

        except Exception as e:
            print(f'Error processing {filename} and {path_output}: {e}')

In [ ]:
#@title ##**Download Results** { display-mode: "form" }
import os, zipfile
from google.colab import files

os.chdir('/content/DiT4SR')

if 'result_folder' not in globals():
    result_folder = 'results/user_upload'

IMG_EXT = {'.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tiff', '.gif'}

output_files = []
for root, _, fnames in os.walk(result_folder):
    for f in fnames:
        if os.path.splitext(f.lower())[1] in IMG_EXT and not f.startswith('.'):
            output_files.append(os.path.join(root, f))
output_files.sort()

if not output_files:
    print(f'Изображений для скачивания не найдено в {result_folder}.')
elif len(output_files) == 1:
    print(f'Скачивание: {os.path.basename(output_files[0])}')
    files.download(output_files[0])
else:
    zip_name = 'DiT4SR_results.zip'
    if os.path.exists(zip_name):
        os.remove(zip_name)
    print(f'Архивирую {len(output_files)} файлов в {zip_name}...')
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fp in output_files:
            zf.write(fp, arcname=os.path.basename(fp))
    files.download(zip_name)
